# 02 — Feature Engineering
## AI-Driven Predictive Monitoring System for Supply Chain Disruptions

This notebook transforms raw data into a rich feature matrix for model training.

**Pipeline Steps:**
1. Load & merge all datasets
2. One-hot encode `transport_mode`
3. Extract temporal features (cyclical sin/cos encoding)
4. Rolling-window port congestion features (7/14/30 days)
5. Lag features (t−1, t−7, t−14 congestion)
6. Interaction features (compound risk terms)
7. Composite risk score
8. Mutual information & feature importance analysis
9. Time-based train/validation/test split
10. Export processed feature matrix


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

ROOT     = Path("..").resolve()
DATA_DIR = ROOT / "data"
PROC_DIR = DATA_DIR / "processed"
PROC_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(ROOT))
from src.data.loader import SupplyChainLoader
from src.features.build_features import (
    encode_transport_mode, add_temporal_features,
    add_port_rolling_features, add_lag_features,
    add_interaction_features, compute_risk_score,
    FEATURE_COLS, TARGET_CLF, TARGET_REG,
)

loader     = SupplyChainLoader(DATA_DIR)
shipments  = loader.shipments()
congestion = loader.port_congestion()

print("Shipments shape:", shipments.shape)
print("Congestion shape:", congestion.shape)


In [ ]:
# ── Step 1: Encode transport mode (OHE) ──────────────────────────────────────
df = encode_transport_mode(shipments)
print("After OHE:", [c for c in df.columns if c.startswith("mode_")])

# ── Step 2: Temporal features ─────────────────────────────────────────────────
df = add_temporal_features(df, date_col="ship_date")
print("Temporal cols:", [c for c in df.columns if c in ["month","day_of_week","is_weekend","month_sin","month_cos"]])

# ── Step 3: Rolling congestion features ──────────────────────────────────────
df = add_port_rolling_features(df, congestion, windows=[7, 14, 30])
roll_cols = [c for c in df.columns if "roll" in c]
print("Rolling cols:", roll_cols)

# ── Step 4: Lag features ──────────────────────────────────────────────────────
df = add_lag_features(df, congestion, lags=[1, 7, 14])
lag_cols = [c for c in df.columns if "lag" in c]
print("Lag cols:", lag_cols)

# ── Step 5: Interaction features ─────────────────────────────────────────────
df = add_interaction_features(df)
int_cols = [c for c in df.columns if c.startswith("feat_")]
print("Interaction cols:", int_cols)

# ── Step 6: Composite risk score ──────────────────────────────────────────────
df = compute_risk_score(df)

print(f"\nFinal feature matrix shape: {df.shape}")
print("NaN counts:\n", df.isnull().sum().sort_values(ascending=False).head(10))


In [ ]:
# ── Mutual Information Feature Importance ────────────────────────────────────
feat_cols_avail = [c for c in FEATURE_COLS if c in df.columns]
df_clean        = df[feat_cols_avail + [TARGET_CLF]].fillna(df.median(numeric_only=True))

X = df_clean[feat_cols_avail].values
y = df_clean[TARGET_CLF].values

mi_scores = mutual_info_classif(X, y, discrete_features=False, random_state=42)
mi_series = pd.Series(mi_scores, index=feat_cols_avail).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, max(6, len(mi_series) * 0.28)))
mi_series.tail(25).plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Mutual Information with Target `delayed` — Top 25 Features", fontweight='bold')
ax.set_xlabel("MI Score (higher = more informative)")
plt.tight_layout()
plt.show()

print("\nTop 10 features by MI:")
print(mi_series.tail(10).round(4).to_string())


In [ ]:
# ── Time-based Train / Validation / Test Split ──────────────────────────────
df_sorted = df.copy()
# Drop non-feature columns
for dropcol in ["expected_delivery_date","actual_delivery_date","ship_date"]:
    if dropcol in df_sorted.columns:
        df_sorted.drop(columns=[dropcol], inplace=True)

df_sorted.fillna(df_sorted.median(numeric_only=True), inplace=True)

n      = len(df_sorted)
val_c  = int(n * 0.70)
test_c = int(n * 0.85)

train = df_sorted.iloc[:val_c].reset_index(drop=True)
val   = df_sorted.iloc[val_c:test_c].reset_index(drop=True)
test  = df_sorted.iloc[test_c:].reset_index(drop=True)

print(f"Train : {len(train):,} rows  ({len(train)/n*100:.0f}%)")
print(f"Val   : {len(val):,} rows  ({len(val)/n*100:.0f}%)")
print(f"Test  : {len(test):,} rows  ({len(test)/n*100:.0f}%)")
print(f"\nFeatures used: {feat_cols_avail}")

# Save processed splits
train.to_csv(PROC_DIR / "train.csv", index=False)
val.to_csv(  PROC_DIR / "val.csv",   index=False)
test.to_csv( PROC_DIR / "test.csv",  index=False)
print(f"\nSaved to {PROC_DIR}/")
